## HATS Data Preview 1 on RSP

This notebook tests access to Data Preview 1 (DP1) data in the HATS format. 

**Goal:** To load a randomized sample of the data, to be used for scale testing within the RSP.

In [13]:
# if not previously installed
%pip install --upgrade lsdb --quiet

Note: you may need to restart the kernel to use updated packages.


In [14]:
import lsdb
import numpy as np
import pandas as pd
from upath import UPath

In [15]:
base_path = UPath("/rubin/lsdb_data/dp1/")
object_collection = lsdb.open_catalog(base_path / "object_collection")

In [16]:
pixel_statistics = object_collection.per_partition_statistics()
counts = pd.to_numeric(pixel_statistics["objectId: row_count"], errors="coerce")
pixel_counts = counts.groupby(level=0).sum()

In [ ]:
partition_indices = []
for percentile in [10, 50, 90]:
    q = np.percentile(pixel_counts, percentile)
    print(f"Percentile: {percentile}, Quartile: {q}")
    index = int(np.argmin(np.abs(pixel_counts - q)))
    closest_value = pixel_counts.iloc[index]
    print(f"Closest value: {closest_value}, partition index: {index}")
    partition_indices.append(index)

In [ ]:
for index in partition_indices:
    print(f"Sampling partition {index} of size {pixel_counts.iloc[index]}")
    %timeit object_collection.sample(index, n=100, seed=10)